In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)

2026-06-29 06:38:53.928846945 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [4]:
print(f"Q1: The embedder returns a vector of 384 numbers. What's the first value (v[0])?")
print(f"Answer: First value (v[0]): {v1[0]}")

Q1: The embedder returns a vector of 384 numbers. What's the first value (v[0])?
Answer: First value (v[0]): -0.02058203437252893


In [11]:
from gitsource import GithubRepositoryDataReader

# Load data
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

# Find the specific page
target_page = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(d for d in documents if d["filename"] == target_page)

# Embed page content
v_page = embed.encode(target_doc["content"])

# Cosine similarity (dot product for normalized vectors)
similarity = v1.dot(v_page)
print(f"Q2: Cosine similarity between the question and the page content?")
print(f"Cosine similarity: {similarity:.4f}")

Q2: Cosine similarity between the question and the page content?
Cosine similarity: 0.3611


In [14]:
from gitsource import chunk_documents
from tqdm.auto import tqdm
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

# Embed all chunks
texts = [chunk["content"] for chunk in chunks]
batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

scores = X.dot(v1)

# Find best match
best_idx = np.argmax(scores)
best_chunk = chunks[best_idx]
print(f'Q3: Which file contains the best matching chunk for the question?')
print(f"Best matching file: {best_chunk['filename']}")
print(f"Score: {scores[best_idx]:.4f}")

  0%|          | 0/6 [00:00<?, ?it/s]

Q3: Which file contains the best matching chunk for the question?
Best matching file: 02-vector-search/lessons/07-sqlitesearch-vector.md
Score: 0.6489


In [24]:
from minsearch import VectorSearch

# Create minsearch VectorSearch index
vindex = VectorSearch()
vindex.fit(X, chunks)

# Query
q2 = 'What metric do we use to evaluate a search engine?'
v2 = embed.encode(q2)

# Search
results = vindex.search(v2, num_results=1)
print(results)

# Get first result
print(f'Q4: What is the filename of the first result for the query "{q2}"?')
print(f"First result filename: {results[0]['filename']}")


[{'start': 0, 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

In [ ]:
q3 = 'How do I store vectors in PostgreSQL?'
v3 = embed.encode(q3)

from minsearch import Index

# Text search
tindex = Index(text_fields=['content'])
tindex.fit(chunks)

# Get results
vector_results = vindex.search(v3, num_results=5)
text_results = tindex.search(q3, num_results=5)

# Get filenames
vector_files = [r["filename"] for r in vector_results]
text_files = [r["filename"] for r in text_results]

# Find files in vector but not in text
vector_only = set(vector_files) - set(text_files)
print(f'Q5: Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?')
print(f"Files in vector results but not in text results: {vector_only}")


Files in vector results but not in text results: {'02-vector-search/lessons/08-pgvector.md'}


In [47]:
# Your original RRF function (unchanged)
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# Query
q4 = 'How do I give the model access to tools?'
v4 = embed.encode(q4)

# Get results
vector_results = vindex.search(v4, num_results=10)
text_results = tindex.search(q4, num_results=10)

# Get hybrid results (without scores)
hybrid_results = rrf([vector_results, text_results], k=60, num_results=5)

# Calculate RRF scores separately for display
rrf_scores = {}
for results in [vector_results, text_results]:
    for rank, doc in enumerate(results):
        key = (doc["filename"], doc["start"])
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (60 + rank)

# Print results with scores
print(f'Q6: What is the filename of the first result after applying RRF for the query "{q4}"?')
for i, doc in enumerate(hybrid_results, 1):
    key = (doc["filename"], doc["start"])
    score = rrf_scores.get(key, 0)
    print(f"{i}. {doc['filename']}")
    print(f"   RRF score: {score:.6f}")
    print()

print(f"First result after RRF: {hybrid_results[0]['filename']}")

Q6: What is the filename of the first result after applying RRF for the query "How do I give the model access to tools?"?
1. 01-agentic-rag/lessons/13-function-calling.md
   RRF score: 0.032018

2. 01-agentic-rag/lessons/13-function-calling.md
   RRF score: 0.030090

3. 01-agentic-rag/lessons/01-intro.md
   RRF score: 0.016667

4. 01-agentic-rag/lessons/14-agentic-loop.md
   RRF score: 0.016667

5. 04-evaluation/lessons/02-ground-truth.md
   RRF score: 0.016393

First result after RRF: 01-agentic-rag/lessons/13-function-calling.md
